In [1]:
def inverse_transform_X(X_scaled, scaler_X):
    """
    Convert scaled model input back to original values.

    Transformation applied during preprocessing:
        log1p -> MinMaxScaler

    Reverse:
        MinMaxScaler^-1 -> expm1
    """

    X_log = scaler_X.inverse_transform(X_scaled)

    X_original = X_log.copy()

    # Only first 5 features were log-transformed
    X_original[:, :5] = np.expm1(
        X_original[:, :5]
    )

    return X_original
def inverse_transform_y(y_scaled, scaler_y):
    """
    Convert scaled target/prediction back to original units.

    Transformation:
        log1p -> MinMaxScaler

    Reverse:
        MinMaxScaler^-1 -> expm1
    """

    y_log = scaler_y.inverse_transform(y_scaled)

    y_original = np.expm1(y_log)

    return y_original
def summarize_captum(
    attribution,
    feature_names,
    top_k=5
):
    """
    Summarize a single target's Integrated Gradients.

    attribution shape:
        (sequence_length, num_features)

    Returns the top positive and negative
    feature/time contributions.
    """

    attribution = np.asarray(attribution)

    sequence_length, num_features = attribution.shape

    # Flatten:
    # [time, feature] -> single vector
    flat = attribution.flatten()

    # Top positive
    positive_indices = np.argsort(flat)[-top_k:][::-1]

    # Top negative
    negative_indices = np.argsort(flat)[:top_k]

    def decode(indices):

        output = []

        for index in indices:

            timestep = index // num_features
            feature_index = index % num_features

            output.append({
                "timestep": int(timestep + 1),
                "feature": feature_names[feature_index],
                "attribution": float(flat[index])
            })

        return output

    return {
        "positive": decode(positive_indices),
        "negative": decode(negative_indices)
    }

In [2]:
import json 
import numpy as np
import mlflow 
import joblib as jl
import pandas as pd
mlflow.set_tracking_uri("http://localhost:5000")
with open("captum.json", "r") as f:
    data = json.load(f)

sample_id = "sample_1"

sample = data["samples"][sample_id]

# get experiment
experiment = mlflow.get_experiment_by_name("Default")

# search runs
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.stage = 'preprocessing'",
    order_by=["start_time DESC"],
    max_results=1
)
run_id = runs.iloc[0].run_id

scaler_path_X = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="data/scaler_X.pkl"
)
scaler_path_y = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="data/scaler_y.pkl"
)

# Load data
scaler_X = jl.load(scaler_path_X)
scaler_y = jl.load(scaler_path_y)

prediction_scaled = np.array(
    list(sample["prediction"].values())
).reshape(-1, 1)

actual_scaled = np.array(
    list(sample["real_value"].values())
).reshape(-1, 1)

prediction_original = inverse_transform_y(
    prediction_scaled,
    scaler_y
)

actual_original = inverse_transform_y(
    actual_scaled,
    scaler_y
)

    # -------------------------------
# Summarize Captum
# -------------------------------
FEATURE_NAMES = [
    "Global_active_power",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
    "year",
    "quarter_sin",
    "quarter_cos",
    "month_sin",
    "month_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "weekday"
]
captum_summary = {}

for target_idx in range(30):

    target_key = f"target_{target_idx + 1}"

    attribution_matrix = np.array([
        list(window.values())
        for window in sample[
            "window_attribution"
        ][target_key].values()
    ])

    captum_summary[target_key] = summarize_captum(
        attribution_matrix,
        FEATURE_NAMES,
        top_k=5
    )

In [3]:
print(attribution_matrix)

[[-3.89117036e+16 -1.56803579e+15 -4.02852299e+16 ...  6.10488541e+16
  -2.55190475e+15 -8.64223699e+16]
 [-1.08766128e+16 -2.04348939e+16 -2.70723488e+17 ... -6.98173434e+16
   1.06349060e+16  6.39886508e+14]
 [ 4.90956047e+15  3.49389791e+15  3.37228828e+16 ...  3.34581388e+16
  -3.73210351e+15  6.81484394e+16]
 ...
 [-1.89382827e-03 -7.56298425e-04  6.15266035e-04 ...  2.15750188e-03
   3.66232242e-04 -2.08928413e-03]
 [ 4.14338056e-03  4.91714384e-03 -5.14205894e-04 ...  2.83384975e-03
   3.55690194e-04 -4.76195570e-03]
 [ 7.28390832e-03  8.05536658e-03 -2.21853121e-03 ...  9.04200878e-03
   2.78227730e-03 -7.25904887e-04]]


In [4]:
prompt = f"""
You are explaining the prediction of a 30-step
multivariate energy forecasting model.

CURRENT SAMPLE
==============

Sample ID: {sample["sample_id"]}

MODEL PREDICTIONS:
{prediction_original}

ACTUAL VALUES:
{actual_original}

ATTRIBUTIONS FEATURES NAMES IN ORDER:-
{FEATURE_NAMES}

INTEGRATED GRADIENTS ATTRIBUTIONS:
{attribution_matrix}

Explain the model's prediction.

Identify the features and historical time steps that
had the strongest influence.

Distinguish between model attribution and causality.
Do not claim that an attribution proves that a feature
caused the prediction.
"""

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import torch
model_name = "Qwen/Qwen3-4B-Instruct-2507"
print(1)
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(1)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    max_memory={
        0: "3GiB",
        "cpu": "12GiB",
    },
)
print(1)
# prepare the model input

messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print(1)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=16384
)
print(1)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print("content:", content)


1


1


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

1
1
1
content: Let's carefully and thoroughly **explain the model's prediction**, **identify the most influential features and historical time steps**, and **distinguish between model attribution and causality** in the context of this 30-step multivariate energy forecasting model.

---

## 🔍 **1. Understanding the Model Prediction**

We are analyzing a **30-step multivariate energy forecasting model** that predicts future energy consumption values using a set of historical features.

### 📌 Model Output (Predictions vs. Actuals)

| Feature | Prediction (Model) | Actual (Ground Truth) |
|--------|--------------------|------------------------|
| General Trend | Starts at ~0.7, gradually rises to ~0.86, then slightly dips and stabilizes | Starts at ~1.4, drops to ~1.3, peaks at ~2.0, then declines to ~0.4 |

> ❗ **Key Observation**:  
> The **model predictions are significantly lower** than the actual values — especially in the early steps.  
> For example:
> - Step 0: Model predicts 0.698

In [7]:
print("Prompt characters:", len(prompt))

prompt_tokens = tokenizer(
    prompt,
    return_tensors="pt"
).input_ids

print("Prompt tokens:", prompt_tokens.shape[1])

Prompt characters: 1930
Prompt tokens: 1388


In [6]:
from pathlib import Path
import json

# Same output directory used by Model_Evaluation.ipynb
OUTPUT_DIR = Path("/workspace/evaluation_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LLM_OUTPUT_PATH = OUTPUT_DIR / "llm_outputs.json"

# Load existing outputs if the file already exists
if LLM_OUTPUT_PATH.exists():
    with open(LLM_OUTPUT_PATH, "r", encoding="utf-8") as f:
        llm_outputs = json.load(f)
else:
    llm_outputs = {}

# Store the current explanation
llm_outputs[str(sample_id)] = content

# Save all explanations
with open(LLM_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(llm_outputs, f, indent=4, ensure_ascii=False)

print(f"LLM output saved to: {LLM_OUTPUT_PATH}")
print(f"Total saved explanations: {len(llm_outputs)}")

LLM output saved to: /workspace/evaluation_outputs/llm_outputs.json
Total saved explanations: 1


In [8]:
import torch
import gc  # Garbage collection module

def clear_vram():
    """
    Safely clears all GPU VRAM resources used by PyTorch in the current process.
    """
    try:
        # Keep essential names so we don't delete them
        keep_vars = {"torch", "gc", "clear_vram", "__name__", "__doc__", "__package__", "__loader__", "__spec__", "__annotations__", "__builtins__"}

        # Delete all other global variables that may hold GPU tensors/models
        for obj in list(globals().keys()):
            if obj not in keep_vars:
                del globals()[obj]

        # Force Python garbage collection
        gc.collect()

        # Clear PyTorch CUDA cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()       # Releases cached memory
            torch.cuda.ipc_collect()       # Collects inter-process memory
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.reset_accumulated_memory_stats()
            print("✅ GPU VRAM cleared successfully.")
        else:
            print("⚠️ No CUDA-enabled GPU detected.")

    except Exception as e:
        print(f"Error while clearing VRAM: {e}")

# Example usage:
if __name__ == "__main__":
    clear_vram()


✅ GPU VRAM cleared successfully.
